## 콘텐츠분쟁해결 RAG 시스템

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

print("UPSTAGE_API_KEY:", "설정됨" if os.getenv("UPSTAGE_API_KEY") else "없음")

UPSTAGE_API_KEY: 설정됨


In [2]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("../data/콘텐츠분쟁해결_사례.pdf")
pages = loader.load()

print(f"페이지 수: {len(pages)}")
print(pages[8].metadata)
print(pages[8].page_content[:300])

C:\Users\user\AppData\Local\Temp\ipykernel_17616\3641439705.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


페이지 수: 109
{'producer': 'Hancom PDF 1.3.0.410', 'creator': 'Hancom PDF 1.3.0.410', 'creationdate': '2011-01-20T18:01:33+09:00', 'title': '제 2절 영국사례', 'moddate': '2011-01-20T18:01:33+09:00', 'pdfversion': '1.4', 'source': '../data/콘텐츠분쟁해결_사례.pdf', 'total_pages': 109, 'page': 8, 'page_label': '9'}
1. 게임
3게임1
■ 1-가-1) 한국소비자원 조정전 상담사례
2006_시스템 오류로 소멸된 아이템 복구 요구
【사건개요 】
신청인은 피신청인이 제공하는 온라인게임서비스를 이용하던 중 시스템 오류로 120,000 원 상당의 
아이템이 소멸되었고 , 이에 피신청인에게 수차례 이의제기하였으나 사과 공지문만 안내 되고 아이템
이 복구 되지 않았다 . 피신청인이 정상적으로 거래하던 중 아이템이 소멸된 것을 확인해 주었으나 
규정상 복구가 불가하다는 답변을 한 것에 대하여 신청인은 정상적으로 이용하다 시스템 오류로 
인해 소멸된 아이템에 


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import statistics

guide_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=300,
    separators=["\n【사건개요】", "\n【쟁점사항】", "\n【처리경위】", "\n【처리결과】",
                "\n■", "\n\n", "\n", ".", " ", ""],
)
guide_splits = guide_splitter.split_documents(pages)

lens = [len(d.page_content) for d in guide_splits]
print(f"[가이드 방식] 청크 {len(guide_splits)}개 | 길이 최소 {min(lens)} / 평균 {int(statistics.mean(lens))} / 최대 {max(lens)}")

full_raw = "\n".join(p.page_content for p in pages)
for sep in ["\n【사건개요】", "\n【쟁점사항】", "\n■"]:
    print(f"{sep!r:>16} 등장 횟수: {full_raw.count(sep)}")

[가이드 방식] 청크 104개 | 길이 최소 32 / 평균 753 / 최대 1442
      '\n【사건개요】' 등장 횟수: 0
      '\n【쟁점사항】' 등장 횟수: 0
           '\n■' 등장 횟수: 3


In [4]:
import re
import bisect
from langchain_core.documents import Document

HEADER_RE = re.compile(r"콘텐츠분쟁조정\s*법리\s*연구\s*제?\s*2\s*부\s*-\s*타\s*분쟁조정사례\s*조사\s*-")
CHAPTER_RE = re.compile(r"([1-4])\s*\.\s*(게임|이러닝|기타\(웹\)|他)")
CHAPTER_NAME = {"1": "게임", "2": "이러닝", "3": "웹콘텐츠", "4": "법령"}

page_texts, page_cats, page_starts = [], [], []
current_cat, pos = "서론", 0
for p in pages:
    text = HEADER_RE.sub("", p.page_content)
    m = CHAPTER_RE.search(text[:80])          
    if m:
        current_cat = CHAPTER_NAME[m.group(1)]
    page_texts.append(text)
    page_cats.append(current_cat)
    page_starts.append(pos)
    pos += len(text) + 1                     

full_text = "\n".join(page_texts)

def page_of(offset: int) -> int:
    """전체 텍스트의 위치(offset) → 페이지 번호(0부터)"""
    return bisect.bisect_right(page_starts, offset) - 1

law_start = page_starts[page_cats.index("법령")]
starts = [m.start() for m in re.finditer(r"(?m)^■", full_text) if m.start() < law_start]
ends = starts[1:] + [law_start]

case_docs = []
for s, e in zip(starts, ends):
    body = full_text[s:e].strip()
    title = body.split("【")[0].lstrip("■ ").strip()    
    pg = page_of(s)
    case_docs.append(Document(
        page_content=body,
        metadata={"category": page_cats[pg], "title": title, "page": pg + 1},
    ))

law_doc = Document(
    page_content=full_text[law_start:page_starts[-1]],
    metadata={"category": "법령", "title": "분쟁조정 사례에 적용된 법", "page": page_of(law_start) + 1},
)

print(f"사례 수: {len(case_docs)}")
print({c: sum(d.metadata["category"] == c for d in case_docs) for c in ["게임", "이러닝", "웹콘텐츠"]})
print(case_docs[0].metadata)

사례 수: 3
{'게임': 1, '이러닝': 1, '웹콘텐츠': 1}
{'category': '게임', 'title': '1-가-1) 한국소비자원 조정전 상담사례\n2006_시스템 오류로 소멸된 아이템 복구 요구', 'page': 9}


In [5]:

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=300,
    separators=["【사건개요】", "【쟁점사항】", "【처리경위】", "【처리결과】", "【관련법령】",
                "\n", ".", " ", ""],
)
splits = text_splitter.split_documents(case_docs + [law_doc])

for d in splits:
    if d.metadata["category"] != "법령" and not d.page_content.startswith("■"):
        d.page_content = f"[사례] {d.metadata['title']}\n{d.page_content}"

lens = [len(d.page_content) for d in splits]
print(f"[개선 방식] 청크 {len(splits)}개 | 길이 최소 {min(lens)} / 평균 {int(statistics.mean(lens))} / 최대 {max(lens)}")

[개선 방식] 청크 60개 | 길이 최소 497 / 평균 1485 / 최대 1554


In [6]:
from langchain_upstage import UpstageEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = UpstageEmbeddings(model="solar-embedding-1-large")

vectorstore = FAISS.from_documents(splits, embeddings)
print(f"벡터 수: {vectorstore.index.ntotal}")

벡터 수: 60


In [7]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)

docs = retriever.invoke("온라인 게임에서 시스템 오류로 아이템이 사라졌는데 복구를 거부합니다")
for d in docs:
    print(f"[{d.metadata['category']} | p.{d.metadata['page']}] {d.metadata['title'][:60]}")

[게임 | p.9] 1-가-1) 한국소비자원 조정전 상담사례
2006_시스템 오류로 소멸된 아이템 복구 요구
[게임 | p.9] 1-가-1) 한국소비자원 조정전 상담사례
2006_시스템 오류로 소멸된 아이템 복구 요구
[게임 | p.9] 1-가-1) 한국소비자원 조정전 상담사례
2006_시스템 오류로 소멸된 아이템 복구 요구
[게임 | p.9] 1-가-1) 한국소비자원 조정전 상담사례
2006_시스템 오류로 소멸된 아이템 복구 요구
[게임 | p.9] 1-가-1) 한국소비자원 조정전 상담사례
2006_시스템 오류로 소멸된 아이템 복구 요구


In [9]:
from langchain_upstage import ChatUpstage
from langchain_core.prompts import PromptTemplate

llm = ChatUpstage(
    model="solar-pro3",
    base_url="https://api.upstage.ai/v1",
    temperature=0.2,
)

prompt_template = """당신은 콘텐츠 분야 전문 법률 자문가입니다.
아래 분쟁조정 사례들을 바탕으로 정확하고 전문적인 법률 조언을 제공해주세요.

관련 분쟁사례:
{context}

상담 내용: {question}

답변 가이드라인:
1. 제시된 사례들을 근거로 답변하세요. (근거 사례의 제목을 함께 적으세요)
2. 관련 법령이나 조항이 있다면 명시하세요.
3. 비슷한 사례의 처리경위와 결과를 참고하여 설명하세요.
4. 실무적 해결방안을 단계별로 제시하세요.
5. 사례에 없는 내용은 "제시된 사례집에서는 확인할 수 없습니다"라고 명시하세요.

전문 법률 조언:"""

prompt = PromptTemplate.from_template(prompt_template)

In [10]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    """검색된 Document 리스트 → 출처가 붙은 하나의 문자열"""
    return "\n\n---\n\n".join(
        f"[분야: {d.metadata['category']} | p.{d.metadata['page']}]\n{d.page_content}"
        for d in docs
    )

qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [11]:
test_questions = [
    "온라인 게임에서 시스템 오류로 아이템이 사라졌는데, 게임회사가 복구를 거부하고 있습니다. 어떻게 해결할 수 있나요?",
    "인터넷 강의를 중도 해지하려고 하는데 과도한 위약금을 요구받고 있습니다. 정당한가요?",
    "무료체험 후 자동으로 유료전환되어 요금이 청구되었습니다. 환불 가능한가요?",
    "미성년자가 부모 동의 없이 게임 아이템을 구매했습니다. 환불받을 수 있는 방법이 있나요?",
    "온라인 교육 서비스가 광고와 다르게 제공되어 계약을 해지하고 싶습니다. 가능한가요?",
]

answers = qa_chain.batch(test_questions)

for i, (q, a) in enumerate(zip(test_questions, answers), 1):
    print("=" * 80)
    print(f"[질문 {i}] {q}")
    print("-" * 80)
    print(a)

[질문 1] 온라인 게임에서 시스템 오류로 아이템이 사라졌는데, 게임회사가 복구를 거부하고 있습니다. 어떻게 해결할 수 있나요?
--------------------------------------------------------------------------------
# 온라인 게임 시스템 오류 아이템 복구 관련 법률 조언

## 1. 관련 사례 분석

### [사례 1] 2006_시스템 오류로 소멸된 아이템 복구 요구
- **쟁점**: 게임 중 시스템 오류로 인한 피해 보상 여부
- **처리경위**: 피신청인은 신청인이 계정 명의자가 아니라며 처리를 거절. 계정 명의가 타인으로 되어 있고, 약관에서 현금거래 및 계정공유를 인정하지 않아 계정 명의자가 아닐 경우 복구 처리 불가
- **처리결과**: 계정 명의자가 직접 복구신청해야 함

### [사례 2] 2009_시스템 오류로 인한 손실 아이템 복구 요구
- **쟁점**: 시스템 오류 발생 여부
- **처리경위**: 렉(시스템 오류)으로 인해 게임머니 1,000억 소실 주장했으나, 당시 동 시간대 유사 불만 접수 없음. 사용 내역 확인 결과 평상시와 다름없이 사용. 통신회선 불안정 또는 PC 오류 가능성 있음
- **처리결과**: 시스템 오류 발생 보기 어려워 신청인의 통신회선/PC 오류 점검 권고

### [사례 3] 2009_해킹으로 손실된 아이템 복구 요청
- **쟁점**: 해킹된 아이템의 복구 여부
- **처리경위**: 피신청인은 해킹피해 방지를 위해 2가지 이상 보안서비스 사용을 의무화하고 있으며, 이를 준수하지 않은 경우 보상 불가
- **처리결과**: 보안서비스 1개만 사용한 상태에서 피해 발생, 보상 대상 아님

## 2. 관련 법령

- **소비자기본법 제68조**: 소비자분쟁조정위원회에 분쟁조정을 신청할 수 있음
- **온라인 디지털콘텐츠산업 발전법 시행령 제22조 제2항**: 온라인콘텐츠 이용조건 표시 의무
- **전자상거래 등에서의 소비자보호에 관한 법률 시행령 제21조**: 청약철

In [12]:
def classify_dispute_type(query: str) -> str:

    q = query.replace(" ", "")

    game_keywords = ["게임", "아이템", "계정", "캐릭터", "레벨", "길드"]
    elearning_keywords = ["강의", "온라인교육", "이러닝", "수강", "화상교육", "학습"]
    web_keywords = ["웹사이트", "무료체험", "자동결제", "구독", "사이트", "유료전환"]

    if any(k in q for k in game_keywords):
        return "게임"
    if any(k in q for k in elearning_keywords):
        return "이러닝"
    if any(k in q for k in web_keywords):
        return "웹콘텐츠"
    return "기타"

expected = ["게임", "이러닝", "웹콘텐츠", "게임", "이러닝"]
for q, exp in zip(test_questions, expected):
    got = classify_dispute_type(q)
    print(f"{'OK  ' if got == exp else 'FAIL'} {got:<5} (기대: {exp}) | {q[:30]}...")

OK   게임    (기대: 게임) | 온라인 게임에서 시스템 오류로 아이템이 사라졌는데, 게...
OK   이러닝   (기대: 이러닝) | 인터넷 강의를 중도 해지하려고 하는데 과도한 위약금을 ...
OK   웹콘텐츠  (기대: 웹콘텐츠) | 무료체험 후 자동으로 유료전환되어 요금이 청구되었습니다...
OK   게임    (기대: 게임) | 미성년자가 부모 동의 없이 게임 아이템을 구매했습니다....
OK   이러닝   (기대: 이러닝) | 온라인 교육 서비스가 광고와 다르게 제공되어 계약을 해...


In [13]:
from langchain_core.runnables import RunnableLambda

def retrieve_by_type(question: str):
    """질문 분야를 분류해 같은 분야 사례만 검색 (기타면 전체 검색)"""
    category = classify_dispute_type(question)
    kwargs = {"k": 5}
    if category != "기타":
        kwargs.update(filter={"category": category}, fetch_k=50)
    return vectorstore.similarity_search(question, **kwargs)

routed_chain = (
    {"context": RunnableLambda(retrieve_by_type) | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

q = test_questions[2]
print("[기본 검색]")
for d in retriever.invoke(q):
    print(f"  {d.metadata['category']:<5} | {d.metadata['title'][:50]}")
print("[분야 필터 검색]")
for d in retrieve_by_type(q):
    print(f"  {d.metadata['category']:<5} | {d.metadata['title'][:50]}")

print("\n" + routed_chain.invoke(q))

[기본 검색]
  웹콘텐츠  | 3-가 한국소비자원 조정결정례
2007_등록 거절된 영화예매권 이행 요구
  웹콘텐츠  | 3-가 한국소비자원 조정결정례
2007_등록 거절된 영화예매권 이행 요구
  웹콘텐츠  | 3-가 한국소비자원 조정결정례
2007_등록 거절된 영화예매권 이행 요구
  웹콘텐츠  | 3-가 한국소비자원 조정결정례
2007_등록 거절된 영화예매권 이행 요구
  게임    | 1-가-1) 한국소비자원 조정전 상담사례
2006_시스템 오류로 소멸된 아이템 복구 요구
[분야 필터 검색]
  웹콘텐츠  | 3-가 한국소비자원 조정결정례
2007_등록 거절된 영화예매권 이행 요구
  웹콘텐츠  | 3-가 한국소비자원 조정결정례
2007_등록 거절된 영화예매권 이행 요구
  웹콘텐츠  | 3-가 한국소비자원 조정결정례
2007_등록 거절된 영화예매권 이행 요구
  웹콘텐츠  | 3-가 한국소비자원 조정결정례
2007_등록 거절된 영화예매권 이행 요구

# 무료체험 후 자동 유료전환 요금 관련 법률 조언

## 1. 관련 사례 분석

### [사례 1] 2010_무료 이벤트 후 자동 소액 결제 요금 환급요구 (한국소비자원 조정결정례)
- **사건 개요**: 음악 다운로드 사이트 '003'의 7일 무료체험이벤트 참여 후 자동 유료전환되어 4개월간 30,800원 결제
- **쟁점**: 계약 체결 시 중요내용 명시·고지의무 위반 여부
- **처리 결과**: 피신청인 1은 4개월 결제금액 30,800원의 50%인 15,400원 지급

### [사례 2] 2006_무료 광고 후 서비스 이용요금 청구 (전자거래분쟁조정위원회 조정전합의 사례)
- **사건 개요**: 사이버몰 무료 핸드폰 이용권 이벤트 참여 후 휴대폰 결제로 유료서비스 이용료 청구
- **처리 결과**: 회원탈퇴 처리 및 서비스 이용사실이 없으므로 결제대금 환불 조치

## 2. 관련 법령

- **약관의 규제에 대한 법률 제3조**: 약관의 작성 및 설명의무
- **소비자기본법에 따른 소비자분쟁해결